In [6]:
import sys
import time
from config.prompts import SYSTEM_PROMPT

sys.path.append('..')

In [2]:
import pandas as pd
from src.llm.llm import generate_summary
import config.prompts as prompts_module

notes = pd.read_csv('../data/raw/clinical_notes.csv')

patient_id = notes['person_id'].iloc[0]
print('Using patient_id:', patient_id)
print('Columns:', notes.columns.tolist())

summary = generate_summary(
    patient_id=patient_id,
    notes_df=notes,
    prompt=prompts_module.SUMMARY_PROMPT
)


Using patient_id: 28570119-9cdc-4120-98c0-4edb76cf36a3
Columns: ['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']


In [3]:
print(summary)

**Longitudinal Clinical Summary for Judith Ada Wells**

**Patient Information:**
- Name: Judith Ada Wells
- Date of Birth: 15/05/84
- NHS Number: 272733208
- Gender: Female

**Major Diagnoses:**
- Reversible Cerebral Vasoconstriction Syndrome (RCVS)

**Clinical Progression:**
- Presented on 07/01/26 with a severe headache after exertion, rated 8/10 in intensity, and was triaged as Category 2 (Urgent).
- Initial observations included BP 160/90 mmHg and HR 88 bpm.
- A brief neurological examination showed no abnormalities.
- CT head scan performed on 07/01/26 at 14:45 showed no evidence of intracranial hemorrhage or mass lesion.
- Blood tests on 07/01/26 at 15:15 revealed mild leukocytosis (WBC 12.5 x 10^9/L) and a CRP of 8 mg/L.
- MRI/MRA arranged to confirm diagnosis and assess for vascular abnormalities.
- Diagnosed with RCVS based on clinical presentation, CT findings, and elevated BP.

**Treatments and Investigations:**
- Started on oral nimodipine 60 mg every 4 hours for suspected 

In [4]:
from data.loader import load_notes, get_patient_ids

notes = load_notes('../data/raw/clinical_notes.csv')
patient_ids = get_patient_ids('../data/patients/longitudinal_patient_ids.json')

In [7]:
results = {}
for patient_id in patient_ids:
    start = time.time()
    summary = generate_summary(patient_id, notes, SYSTEM_PROMPT)
    latency = time.time() - start
    
    results[patient_id] = {
        "summary": summary,
        "latency_seconds": round(latency, 2)
    }
    print(f"Done: {patient_id} ({latency:.1f}s)")


Done: 04df53ea-55c1-48d9-84a1-1f15c133b29b (2.9s)
Done: 137b8481-4f1d-4b7f-babd-20f7117023ad (29.2s)
Done: 359014a1-10e6-4bd8-9ba7-513d021c971e (48.7s)
Done: 5e434d78-b2f6-4d88-b327-fff6ee50b901 (48.2s)
Done: 61699d6d-904a-4ece-9026-cd61b3fd9a50 (46.0s)
Done: 69bf7e25-abb2-4dde-857f-f1138d4d0d8a (42.3s)
Done: 8448b3fa-1f5e-45b5-bda5-eb0b7921b8cc (50.8s)
Done: c50e236f-6b3d-41c8-9e16-7ec343cac820 (45.1s)
Done: c6c45c39-cd73-49dd-818d-0a7865fe8a7f (46.4s)
Done: ff8c4724-b7de-4189-bccf-cffddd4d6d44 (47.5s)


In [10]:
import json

with open('../results/workflow1_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("All done. Results saved.")

All done. Results saved.


In [11]:
# Pick one patient and verify note count
test_id = patient_ids[0]
patient_notes = notes[notes['person_id'] == test_id]
print(f"Total notes: {len(patient_notes)}")
print(f"Admissions: {patient_notes['admission_id'].nunique()}")

Total notes: 66
Admissions: 2
